# 01 - AOI & boundaries (Phase 1)

**Colombo UHI practicum.** Builds and visualises every study-area geometry:
Colombo District + Western Province (FAO GAUL), DS/GN divisions (user assets,
filtered to Colombo District), the CMC boundary (~37 km2 sanity check), the
GHSL-derived urban extent, the combined water mask, and BOTH SUHII
rural-reference definitions (`buffer_ring`, `lcz_based`).

Run top-to-bottom in **Google Colab** after `00_setup_and_auth.ipynb` has
worked once. All logic lives in `src/colombo_uhi/`; this notebook only
orchestrates and displays.

> **Caveat (CLAUDE.md #1):** everything in this project is LAND SURFACE
> TEMPERATURE analysis - never air temperature. The exact caveat string is
> printed from `params["caveats"]` below and must accompany every product.

In [ ]:
# COLAB: RUN THIS CELL
# Clone the repo on first run; fast-forward pull on later runs.
import os

REPO_URL = "https://github.com/Dineth0627/colombo_uhi.git"
REPO_DIR = "/content/" + REPO_URL.rstrip("/").removesuffix(".git").rsplit("/", 1)[-1]

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git pull --ff-only

In [ ]:
# COLAB: RUN THIS CELL  (skip if you already ran notebook 00 in this runtime)
%pip install -q -r requirements.txt
print("\nIf Colab asked to RESTART the runtime: Runtime > Restart session,")
print("then re-run this notebook FROM THE CLONE CELL (skip this pip cell).")

In [ ]:
# COLAB: RUN THIS CELL
# Load params (single source of truth) and initialise Earth Engine.
import sys

sys.path.insert(0, os.path.abspath("src"))
from colombo_uhi import load_params
from colombo_uhi.auth import init_ee

params = load_params()
project = init_ee()
print("Earth Engine initialised with project:", project)
print()
print("CAVEAT:", params["caveats"]["lst_not_air_temp"])

## Step 1 - inspect the uploaded boundary assets

The DS/GN assets are **your uploads**, so their attribute schema depends on the
source: OCHA COD-AB uses `ADM3_EN` / `ADM2_EN`, geoBoundaries uses `shapeName`
and has no parent-district column at all. This cell prints what each asset
actually contains; `aoi._resolve_property()` then picks the right field from the
candidate lists in `params.yaml` (`aoi.assets.*_candidates`).

If a later cell complains that no candidate matched, copy the correct field name
from this output into the matching candidate list.

In [ ]:
# COLAB: RUN THIS CELL
from colombo_uhi import aoi

for label, asset_key in (("DS (admin3)", "ds_divisions"), ("GN (admin4)", "gn_divisions")):
    asset_id = params["aoi"]["assets"][asset_key]
    if not asset_id:
        print(f"{label}: no asset configured (aoi.assets.{asset_key} is null)\n")
        continue
    info = aoi.describe_asset(asset_id, n_samples=2)
    print(f"{label}  ->  {info['asset_id']}")
    print(f"  features (nationwide): {info['count']}")
    print(f"  properties: {info['properties']}")
    for sample in info["samples"]:
        trimmed = {k: v for k, v in sample.items() if k != "system:index"}
        print(f"  sample: {trimmed}")
    print()

In [ ]:
# COLAB: RUN THIS CELL
# Administrative boundaries. DS/GN are filtered to Colombo District, so the
# counts must now be 13 and 557 - NOT the nationwide 339 / 14043.
district = aoi.colombo_district(params)
province = aoi.western_province(params)
print("District features (expect 1):", district.size().getInfo())
print("Province features (expect 3):", province.size().getInfo())
print("Province districts:",
      province.aggregate_array(params["aoi"]["gaul"]["district_property"]).getInfo())

counts = params["aoi"]["expected_counts"]
ds_fc = aoi.ds_divisions(params)
gn_fc = aoi.gn_divisions(params)
for label, fc, expected_n in (("DS", ds_fc, counts["ds_divisions"]),
                              ("GN", gn_fc, counts["gn_divisions"])):
    got = fc.size().getInfo()
    verdict = "OK" if got == expected_n else f"<< CHECK (expected {expected_n})"
    print(f"\n{label} features in Colombo District: {got}  {verdict}")

# The 13 DS names - the ones the CMC is dissolved from must appear here.
assets_cfg = params["aoi"]["assets"]
ds_name_prop = assets_cfg["ds_name_property_candidates"][0]
print("DS names:", sorted(ds_fc.aggregate_array(ds_name_prop).getInfo()))

cmc_geom = None
try:
    cmc_geom = aoi.cmc_boundary(params)
    # Independent cross-check: the asset carries its own area_sqkm, so compare it
    # against the geometry area computed in the next cell.
    cmc_names = params["aoi"]["cmc"]["ds_division_names"]
    import ee
    cmc_parts = ds_fc.filter(ee.Filter.inList(ds_name_prop, cmc_names))
    stated = cmc_parts.aggregate_sum(assets_cfg["area_property"]).getInfo()
    print(f"\nCMC boundary built from {cmc_names}")
    print(f"  sum of the asset's own {assets_cfg['area_property']}: {stated:.2f} km2")
except (RuntimeError, ValueError) as err:
    print("\nCMC boundary FAILED -", err)

In [ ]:
# COLAB: RUN THIS CELL
# Area sanity checks. GAUL is simplified at 500 m - a few % deviation is normal.
# The urban-extent vectorisation makes this cell take ~1 minute.
expected = params["aoi"]["expected_areas_km2"]

urban_geom = aoi.urban_extent(params)
ring_geom = aoi.buffer_ring(params) if cmc_geom is not None else None

rows = [
    ("Colombo District", district.geometry(10), expected["district"]),
    ("Western Province", province.geometry(10), expected["western_province"]),
    ("CMC", cmc_geom, expected["cmc"]),
    ("Urban extent (GHSL)", urban_geom, None),
    ("Rural buffer ring", ring_geom, None),
]
print(f"{'AOI':<22}{'area km2':>12}{'expected':>12}")
for name, geom, exp in rows:
    if geom is None:
        print(f"{name:<22}{'- unavailable':>12}{exp or '':>12}")
        continue
    km2 = aoi.area_km2(geom).getInfo()
    flag = ""
    if exp:
        flag = "  OK" if abs(km2 - exp) / exp <= 0.10 else "  << CHECK (>10% off)"
    print(f"{name:<22}{km2:>12.1f}{exp or '':>12}{flag}")

In [ ]:
# COLAB: RUN THIS CELL
# Combined water mask: MNDWI OR QA_PIXEL-water-frequency OR JRC occurrence.
# Plus a 60 m shoreline-buffer variant (coastal mixed-pixel exclusion demo;
# the project default aoi.water_mask.shoreline_buffer_m is 0 = off).
water = aoi.water_mask(params)
water_buffered = aoi.water_exclusion_mask(params, shoreline_buffer_m=60)
print("Water mask built. Threshold config:", params["aoi"]["water_mask"])

In [ ]:
# COLAB: RUN THIS CELL
# BOTH SUHII rural-reference definitions behind the common interface.
# Later phases flip between them with this one string (CLAUDE.md caveat 5:
# always report the sensitivity, never a single number).
suhii = params["uhi"]["suhii"]
urban_lcz, rural_lcz = aoi.rural_reference("lcz_based", params)

# buffer_ring is based on the CMC, so it is skipped rather than allowed to kill
# the rest of the notebook when the DS name match failed above.
urban_br = rural_br = None
if cmc_geom is not None:
    urban_br, rural_br = aoi.rural_reference("buffer_ring", params)
else:
    print("SKIPPED buffer_ring rural reference - no CMC boundary.\n")

print("buffer_ring: ring", suhii["buffer_ring"]["inner_km"], "-",
      suhii["buffer_ring"]["outer_km"], "km beyond the",
      suhii["buffer_ring"]["base"], "; excludes", suhii["buffer_ring"]["exclude"])
print("lcz_based: urban =", suhii["lcz_based"]["urban_classes"],
      "| rural =", suhii["lcz_based"]["rural_classes"],
      "(A-G; water/class G removed by the water mask)")
print("lcz_based scope:", suhii["lcz_based"]["scope"],
      "- both LCZ masks are clipped to this geometry")
print("rural elevation cap:", suhii["rural_filters"]["max_elevation_m"], "m")
print()
print("The two definitions differ by design: buffer = CMC vs a 15-25 km ring;")
print("LCZ = built vs vegetated INSIDE the district. The LCZ rural reference sits")
print("closer to the core, so advection may damp its SUHII - report both.")

In [ ]:
# COLAB: RUN THIS CELL
# Mask areas - the guard against a rural reference the elevation cap has
# emptied. Reduced at 300 m for speed; these are approximate by design.
print(f"{'mask':<34}{'area km2':>12}")
for label, mask in (
    ("urban - buffer_ring (CMC)", urban_br),
    ("rural - buffer_ring", rural_br),
    ("urban - lcz_based (LCZ 1-10)", urban_lcz),
    ("rural - lcz_based (LCZ A-G)", rural_lcz),
    ("water mask", water),
):
    if mask is None:
        print(f"{label:<34}{'- skipped':>12}")
        continue
    km2 = aoi.mask_area_km2(mask, params, scale_m=300).getInfo()
    flag = "  << CHECK (near-empty)" if km2 < 5 else ""
    print(f"{label:<34}{km2:>12.1f}{flag}")

In [ ]:
# COLAB: RUN THIS CELL
# Static PNGs into figures/ - the interactive map below renders nothing once
# this notebook is saved, so these are the shareable verification evidence.
import ee
from IPython.display import Image, display

from colombo_uhi import viz

district_region = district.geometry(10).bounds(10)
province_region = aoi.analysis_region(params).bounds(10)
# Central Colombo, ~10 km around the city centre. The district-wide figure is
# ~50 m/px, too coarse to resolve Beira Lake (0.65 km2) or Diyawanna Lake; this
# zoom is ~22 m/px and does.
centre_pt = ee.Geometry.Point([params["aoi"]["centre"]["lon"],
                               params["aoi"]["centre"]["lat"]])
core_region = centre_pt.buffer(10000).bounds(10)
backdrop = viz.elevation_backdrop(params)

figures = {
    "aoi_boundaries.png": (
        province_region,
        [
            backdrop,
            viz.outline_image(province, "000000", 2),
            viz.outline_image(district, "d62728", 2),
            viz.outline_image(urban_geom, "ff7f0e", 2),
        ]
        + ([viz.outline_image(cmc_geom, "9467bd", 3)] if cmc_geom is not None else []),
    ),
    "aoi_water_mask.png": (
        district_region,
        [
            backdrop,
            water.selfMask().visualize(palette=["1f77b4"]),
            viz.outline_image(district, "d62728", 2),
        ],
    ),
    # Zoomed: Beira Lake, Diyawanna (Parliament) Lake, Kelani River mouth.
    "aoi_water_mask_core.png": (
        core_region,
        [
            backdrop,
            water.selfMask().visualize(palette=["1f77b4"]),
        ]
        + ([viz.outline_image(cmc_geom, "9467bd", 2)] if cmc_geom is not None else []),
    ),
    "aoi_rural_lcz.png": (
        district_region,
        [
            backdrop,
            rural_lcz.selfMask().visualize(palette=["2ca02c"]),
            urban_lcz.selfMask().visualize(palette=["d62728"]),
            viz.outline_image(district, "000000", 2),
        ],
    ),
}
if rural_br is not None:
    figures["aoi_rural_buffer_ring.png"] = (
        province_region,
        [
            backdrop,
            rural_br.selfMask().visualize(palette=["2ca02c"]),
            urban_br.selfMask().visualize(palette=["d62728"]),
            viz.outline_image(district, "000000", 1),
        ],
    )

for filename, (region, layers) in figures.items():
    path = viz.save_thumbnail(layers, region, os.path.join("figures", filename))
    print(path, "-", path.stat().st_size // 1024, "KB")
    display(Image(filename=str(path)))

In [ ]:
# COLAB: RUN THIS CELL
# Interactive map for zooming around. Toggle layers in the layer control.
# (Its output does NOT persist when the notebook is saved - see figures/ above.)
import geemap

centre = params["aoi"]["centre"]
m = geemap.Map(center=[centre["lat"], centre["lon"]], zoom=10)

m.addLayer(viz.outline_image(province, "000000"), {}, "Western Province (GAUL)")
m.addLayer(viz.outline_image(district, "d62728"), {}, "Colombo District (GAUL)")
m.addLayer(viz.outline_image(ds_fc, "8c564b", 1), {}, "DS divisions", False)
m.addLayer(viz.outline_image(gn_fc, "c49c94", 1), {}, "GN divisions", False)
if cmc_geom is not None:
    m.addLayer(viz.outline_image(cmc_geom, "9467bd", 3), {}, "CMC (from DS asset)")
m.addLayer(viz.outline_image(urban_geom, "ff7f0e"), {}, "Urban extent (GHSL)", False)
if ring_geom is not None:
    m.addLayer(viz.outline_image(ring_geom, "2ca02c"), {}, "Rural buffer ring (15-25 km)")

m.addLayer(water.selfMask(), {"palette": ["1f77b4"]}, "Water mask")
m.addLayer(water_buffered.selfMask(), {"palette": ["17becf"]},
           "Water mask + 60 m shoreline buffer", False)
if rural_br is not None:
    m.addLayer(rural_br.selfMask(), {"palette": ["98df8a"]},
               "Rural mask - buffer_ring method", False)
m.addLayer(urban_lcz.selfMask(), {"palette": ["7f7f7f"]},
           "Urban mask - LCZ classes 1-10", False)
m.addLayer(rural_lcz.selfMask(), {"palette": ["2ca02c"]},
           "Rural mask - LCZ A-G minus water", False)
m

## Visual verification checklist (Phase 1 sign-off)

Counts and areas:

- [ ] District = **1** feature, ~**686 km2** (expected 699, GAUL-simplified).
- [ ] Western Province = **3** features (Colombo, Gampaha, Kalutara), ~**3762 km2**.
- [ ] DS divisions in Colombo District = **13** (not the nationwide 339).
- [ ] GN divisions in Colombo District = **557** (not the nationwide 14043).
- [ ] The printed DS-name list contains the two names in
      `aoi.cmc.ds_division_names`; if not, paste the right spellings in.
- [ ] CMC area ~ **37 km2**, and close to the asset's own `area_sqkm` sum
      printed alongside it (Colombo DS alone is 24.54 km2 in the asset).
- [ ] Rural buffer ring is a compact band around the CMC (it was 4049 km2 when
      based on the province-wide GHSL extent).
- [ ] LCZ urban is now a Colombo-District-scale number, far below the
      region-wide **2464 km2** of the previous run, with a non-empty LCZ rural
      counterpart surviving the 100 m elevation cap.
- [ ] No mask flagged near-empty.

In `figures/` (and inline above):

- [ ] `aoi_water_mask_core.png` (zoomed, ~22 m/px) - **Beira Lake** and
      **Diyawanna (Parliament) Lake** are both visible, plus the **Kelani River**
      mouth. The district-wide figure is too coarse to resolve these.
- [ ] `aoi_water_mask.png` - ocean, **Bolgoda Lake**, the Kelani and the
      Labugama/Kalatuwawa reservoirs in the east all captured.
- [ ] `aoi_boundaries.png` - CMC (purple) sits inside the district (red), which
      sits inside the province (black).
- [ ] `aoi_rural_lcz.png` - red (LCZ built) over Colombo and its suburbs, green
      (LCZ A-G) in the district's eastern/southern parts, nothing outside the
      district outline, no green over water.
- [ ] `aoi_rural_buffer_ring.png` - red core = CMC, green ring 15-25 km outside
      it, no green over water or high ground.

Use the interactive map to zoom into each water body. **Send the PNGs back for
review before Phase 2.**

**Next:** Phase 2 - `02_lst_pipeline.ipynb` (harmonised Landsat LST + MODIS).

## Appendix - re-uploading the DS/GN boundary assets

GAUL stops at district level for Sri Lanka, so DS divisions, GN divisions and
the CMC boundary need **user-uploaded EE assets** (see PROGRESS.md, 2026-08-08).

1. Download **"Sri Lanka - Subnational Administrative Boundaries"** from
   OCHA/HDX (<https://data.humdata.org/dataset/cod-ab-lka>): the **admin3**
   shapefile = DS divisions, **admin4** = GN divisions. Check the licence before
   redistributing derived maps.
2. In the EE Code Editor (<https://code.earthengine.google.com>):
   *Assets > New > Shape files* - upload each shapefile set
   (`.shp .shx .dbf .prj` together).
3. Put the asset ids into `config/params.yaml` under `aoi.assets.ds_divisions`
   / `aoi.assets.gn_divisions`, commit, push, re-run.
4. The uploaded layers cover **all of Sri Lanka**; `aoi.ds_divisions()` and
   `aoi.gn_divisions()` filter to Colombo District for you - by attribute when
   the asset has a parent-district column, otherwise by a
   centroid-within-district spatial test (geoBoundaries has no such column).